# PS1 - can the OOS label be sharpened from silver?

The current PS1 event is "an OOS Set episode". That treats a failed gate and a parked bus
validator identically, which is the most likely reason VALIDATOR scores AUC 0.997 (persistence,
not prediction) while GATE and TVM sit near 0.89.

Silver holds several things that could separate a genuine service-affecting failure from an
operational one. This notebook measures which of them are actually reachable and what each does
to the base rate - and therefore to the AUC the contract demands.

**Read-only.** Reads the PS3 episode fact and the `chicago/silver/` parquet exports. No Aurora,
no Spark, no writes. Big tables are inspected by parquet footer metadata only, never loaded.

Candidates, in the order the repo suggests they matter:

| # | Source | What it adds |
|---|---|---|
| E1 | `dim_event_matrix.oos_counted_{gate,bus,fmvd}_kpi` | **Cubic's own rule** for whether an OOS counts toward that fleet's Ventra availability KPI |
| E2 | `dim_event_matrix.event_priority` | 1=critical .. 4=info |
| E3 | `device_outage` / `device_failures` | `is_chargeable`, `failure_level > 0` (TVM/GATE only - see notes) |
| E4 | `kpi_avail_enriched` relief columns | episodes Cubic itself excused |
| E5 | `maintenance_ledger` | commanded-OOS / maintenance-mode / tech-login windows to exclude |

In [ ]:
# CELL 1 : configuration
import pandas as pd, numpy as np, s3fs, pyarrow.parquet as pq
from math import sqrt
from statistics import NormalDist

BUCKET        = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
EPISODE_PREFIX = "chicago/ps3_outputs/ps3_device_episode_fact"
SILVER_PREFIX  = "chicago/silver"

ELIGIBLE_DEVICES = {"GATE": 821, "TVM": 468, "VALIDATOR": 3028}
MEASURED_AUC     = {"GATE": 0.888, "TVM": 0.894, "VALIDATOR": 0.997}
N = NormalDist()

def auc_required(p, precision=0.90, recall=0.80):
    if not (0 < p < 1):
        return np.nan, np.nan
    fpr = recall * p * (1 - precision) / (precision * (1 - p))
    if fpr >= 1:
        return fpr, 0.5
    return fpr, N.cdf((N.inv_cdf(recall) - N.inv_cdf(fpr)) / sqrt(2))

fs = s3fs.S3FileSystem()
print("config loaded")

In [ ]:
# CELL 2 : helpers -- read small tables, inspect big ones without loading
def parts_of(prefix):
    """Parquet part files under a prefix (newest computed_date if partitioned)."""
    p = sorted(fs.glob(f"{prefix}/**.parquet"))
    if not p:
        return []
    if any("computed_date=" in x for x in p):
        newest = sorted({x.split("computed_date=")[1].split("/")[0]
                         for x in p if "computed_date=" in x})[-1]
        p = [x for x in p if f"computed_date={newest}" in x]
    return p

def footer(prefix):
    """(rows, columns) from parquet metadata -- no data is read."""
    p = parts_of(prefix)
    if not p:
        return None, []
    rows, cols = 0, []
    for f in p:
        with fs.open(f, "rb") as fh:
            m = pq.ParquetFile(fh)
            rows += m.metadata.num_rows
            if not cols:
                cols = list(m.schema_arrow.names)
    return rows, cols

def load(prefix, columns=None, max_rows=8_000_000):
    """Read a table into pandas, refusing anything oversized."""
    rows, _ = footer(prefix)
    if rows is None:
        raise FileNotFoundError(prefix)
    if rows > max_rows and columns is None:
        raise MemoryError(f"{prefix}: {rows:,} rows -- pass columns= to narrow")
    frames = []
    for f in parts_of(prefix):
        with fs.open(f, "rb") as fh:         # handle, not path: no hive inference
            frames.append(pd.read_parquet(fh, columns=columns))
    return pd.concat(frames, ignore_index=True)

print("helpers ready")

In [ ]:
# CELL 3 : what does the episode fact actually carry?
ep = load(f"{BUCKET}/{EPISODE_PREFIX}")
ep["episode_start"] = pd.to_datetime(ep["episode_start"])
WIN_DAYS = max(1, (ep["episode_start"].max() - ep["episode_start"].min()).days + 1)

print(f"episodes = {len(ep):,}   window {ep['episode_start'].min().date()} "
      f"-> {ep['episode_start'].max().date()}  ({WIN_DAYS} days)\n")
print("COLUMNS:")
for c in sorted(ep.columns):
    nn = ep[c].notna().mean()
    print(f"  {c:42s} {str(ep[c].dtype):12s} non-null {nn:6.1%}")

# the columns that decide which enrichments can be done in pandas alone
KEYS = ["event_code_id", "EVENT_TYPE_ID", "event_type_id", "event_priority",
        "oos_counted_gate_kpi", "oos_counted_bus_kpi", "oos_counted_fmvd_kpi",
        "is_chargeable", "failure_level", "is_commanded_oos_event",
        "component_subsystem", "device_id", "transit_day", "availability_event_id"]
print("\nJOIN / FILTER KEYS PRESENT:")
for k in KEYS:
    print(f"  {'YES' if k in ep.columns else ' - ':4s} {k}")

In [ ]:
# CELL 4 : inventory the silver exports (metadata only -- nothing large is loaded)
CANDIDATES = [
    "dim_event_matrix", "dim_event_type", "dim_device", "dim_failure_level",
    "device_failures", "device_outage", "device_event_enriched",
    "kpi_avail_enriched", "kpi_daily", "maintenance_ledger",
    "incident_history", "incident_root_cause", "incident_task_ci_link",
    "device_mttr", "device_survival_intervals", "warnings_daily",
]
inv = []
for t in CANDIDATES:
    try:
        rows, cols = footer(f"{BUCKET}/{SILVER_PREFIX}/{t}")
        inv.append(dict(table=t, rows=rows, n_cols=len(cols),
                        pandas_ok="yes" if (rows or 0) <= 8_000_000 else "NO - too big",
                        cols=cols))
    except Exception as exc:
        inv.append(dict(table=t, rows=None, n_cols=0, pandas_ok=f"MISSING ({type(exc).__name__})", cols=[]))

print(pd.DataFrame(inv)[["table", "rows", "n_cols", "pandas_ok"]].to_string(index=False))

print("\n--- columns of the small decisive tables ---")
for t in ("dim_event_matrix", "device_failures", "kpi_avail_enriched", "maintenance_ledger"):
    row = next((r for r in inv if r["table"] == t), None)
    if row and row["cols"]:
        print(f"\n{t} ({row['rows']:,} rows):")
        print("   " + ", ".join(row["cols"]))

In [ ]:
# CELL 5 : the event matrix -- Cubic's own OOS counting rule
mtx = load(f"{BUCKET}/{SILVER_PREFIX}/dim_event_matrix")
print(f"dim_event_matrix: {len(mtx):,} event codes\n")

flagcols = [c for c in mtx.columns
            if c.startswith("oos_counted") or c in
            ("is_oos", "is_commanded_oos", "requires_service_call", "event_priority",
             "is_reader_event")]
for c in flagcols:
    s = mtx[c]
    if s.dtype == bool or set(s.dropna().unique()) <= {True, False, 0, 1}:
        print(f"  {c:28s} TRUE on {int(s.fillna(False).astype(bool).sum()):4d} / {len(mtx)} codes")
    else:
        print(f"  {c:28s} {dict(s.value_counts().sort_index())}")

# How much would the KPI rule actually cut? Only answerable if the episode fact
# carries an event code; cell 3 says whether it does.
CODE_COL = next((c for c in ("event_code_id", "EVENT_TYPE_ID", "event_type_id")
                 if c in ep.columns), None)
print(f"\nepisode-fact event code column: {CODE_COL or 'NONE -- E1/E2 need a Databricks-side join'}")

In [ ]:
# CELL 6 : measure the label under each feasible enrichment
KPI_FOR = {"GATE": "oos_counted_gate_kpi", "TVM": "oos_counted_fmvd_kpi",
           "VALIDATOR": "oos_counted_bus_kpi"}

dur = ep.get("oos_minutes_union")
if dur is None or dur.isna().all():
    dur = ep.get("set_signal_span_minutes")
ep["_dur"] = pd.to_numeric(dur, errors="coerce").fillna(0.0)

variants = {"E0  every OOS Set episode (today)": lambda d, f: pd.Series(True, index=d.index),
            "B   duration >= 60 min (study pick)": lambda d, f: d["_dur"] >= 60}

if CODE_COL:
    m = mtx.set_index("event_code_id") if "event_code_id" in mtx.columns else mtx.set_index(mtx.columns[0])
    for col in ("event_priority", *[c for c in mtx.columns if c.startswith("oos_counted")]):
        if col in m.columns:
            ep[f"_mx_{col}"] = ep[CODE_COL].map(m[col])
    def kpi_counted(d, fleet):
        c = f"_mx_{KPI_FOR[fleet]}"
        return d[c].fillna(False).astype(bool) if c in d.columns else pd.Series(False, index=d.index)
    variants["E1  counts toward Ventra KPI"] = kpi_counted
    if "_mx_event_priority" in ep.columns:
        variants["E2  event_priority <= 2"] = lambda d, f: pd.to_numeric(
            d["_mx_event_priority"], errors="coerce").le(2).fillna(False)
        variants["E1+E2  KPI-counted AND priority<=2"] = lambda d, f: kpi_counted(d, f) & pd.to_numeric(
            d["_mx_event_priority"], errors="coerce").le(2).fillna(False)

for c in ("is_chargeable", "failure_level"):
    if c in ep.columns:
        if c == "is_chargeable":
            variants["E3  is_chargeable"] = lambda d, f: d["is_chargeable"].fillna(False).astype(bool)
        else:
            variants["E3  failure_level > 0"] = lambda d, f: pd.to_numeric(
                d["failure_level"], errors="coerce").gt(0).fillna(False)
if "is_commanded_oos_event" in ep.columns:
    variants["E5  NOT commanded/maintenance"] = lambda d, f: ~d["is_commanded_oos_event"].fillna(False).astype(bool)

rows = []
for fleet, g in ep.groupby("mars_device_category"):
    ndev = ELIGIBLE_DEVICES.get(fleet)
    if not ndev:
        continue
    dd = ndev * WIN_DAYS
    for name, pred in variants.items():
        try:
            sub = g[pred(g, fleet)]
        except Exception as exc:
            rows.append(dict(fleet=fleet, variant=name, episodes=np.nan,
                             note=f"{type(exc).__name__}")); continue
        for h in (3, 7):
            p = min(0.999, len(sub) * h / dd)
            fpr, auc = auc_required(p)
            rows.append(dict(fleet=fleet, variant=name, episodes=len(sub),
                             kept=len(sub) / max(1, len(g)), horizon=h, base_rate=p,
                             auc_required=auc, auc_today=MEASURED_AUC[fleet],
                             gap=auc - MEASURED_AUC[fleet], note=""))

res = pd.DataFrame(rows)
pd.set_option("display.width", 210); pd.set_option("display.max_rows", 300)
for fleet in sorted(res["fleet"].dropna().unique()):
    print("=" * 108)
    print(f"{fleet}   eligible={ELIGIBLE_DEVICES[fleet]:,}  device-days={ELIGIBLE_DEVICES[fleet]*WIN_DAYS:,}"
          f"  AUC today={MEASURED_AUC[fleet]}")
    print("=" * 108)
    f = res[(res["fleet"] == fleet) & res["horizon"].notna()]
    print(f[["variant", "episodes", "kept", "horizon", "base_rate",
             "auc_required", "gap"]].round(4).to_string(index=False))
    print()
res.to_csv("ps1_label_enrichment_probe.csv", index=False)
print("saved -> ps1_label_enrichment_probe.csv")

In [ ]:
# CELL 7 : verdict
print("WHAT IS REACHABLE FROM THE NOTEBOOK TODAY\n")
ok  = [r for r in inv if isinstance(r["rows"], int) and r["rows"] <= 8_000_000]
big = [r for r in inv if isinstance(r["rows"], int) and r["rows"] > 8_000_000]
gone = [r for r in inv if not isinstance(r["rows"], int)]
print(f"  usable in pandas      : {len(ok):2d}  {', '.join(r['table'] for r in ok)}")
print(f"  too big (need Spark)  : {len(big):2d}  {', '.join(r['table'] for r in big)}")
print(f"  missing from S3       : {len(gone):2d}  {', '.join(r['table'] for r in gone)}")

d = res[res["horizon"] == 7].dropna(subset=["gap"])
if len(d):
    print("\nSMALLEST AUC GAP PER FLEET at a 7-day horizon:")
    print(d.sort_values("gap").groupby("fleet").head(2)
           [["fleet", "variant", "episodes", "base_rate", "auc_required", "gap"]]
           .round(4).to_string(index=False))
print("\nReminder: a SMALLER base rate RAISES the required AUC. An enrichment that")
print("makes the label more honest will usually make the contract harder, not easier.")
print("The reason to do it anyway is that it is the same event Cubic measures.")